# Titanic — merge explicado uma etapa por vez

Este notebook adapta os **seis exercícios e o desafio final** do material enviado para uma gravação com iniciantes. Cada etapa apresenta objetivo, explicação, código pequeno e interpretação. As resoluções aparecem junto dos enunciados, com resultados já executados.

**Como gravar:** leia a pergunta, explique a chave, peça uma previsão, execute a próxima célula e comente a saída. Oriente os alunos a pausar o vídeo. Execute de cima para baixo; após reiniciar o kernel, execute desde o início.

**Arquivos:** mantenha este notebook, `passageiros.csv`, `viagens.csv`, `resultados.csv` e `portos.csv` na mesma pasta. Não é necessário ler `train(1).csv`: os quatro arquivos já contêm os dados usados na atividade. Abra a pasta no VS Code e selecione um kernel com pandas. Se faltar a biblioteca, use `%pip install pandas` em uma célula e reinicie o kernel.

O material original tinha uma variável `pasta` sem definição na leitura dos arquivos. Aqui cada caminho é explícito, com os arquivos ao lado do notebook.

**Sugestão de gravação:** bloco 1: leitura, conceitos e exercícios 1–2; bloco 2: resumos e exercícios 3–4; bloco 3: auditoria, duplicações e desafio. Para explicar tudo com calma, reserve cerca de 90–120 minutos. Uma introdução de uma hora pode priorizar exercícios 1, 2 e 6.

## O que é merge?
“Merge é uma junção entre tabelas. Eu tenho informações sobre a mesma pessoa em arquivos separados e quero reuni-las. Para isso, preciso de uma chave que diga quais linhas correspondem.”

| Arquivo | Uma linha representa | Chave | Informações principais |
|---|---|---|---|
| passageiros.csv | Um passageiro do conjunto fornecido | PassengerId | Nome, sexo, idade, familiares |
| viagens.csv | Uma viagem por passageiro neste exercício | PassengerId | Classe, bilhete, tarifa e porto |
| resultados.csv | Um resultado por passageiro | PassengerId | Sobrevivência: 0 ou 1 |
| portos.csv | Um porto cadastrado | codigo_porto | Nome do porto |

Não use nome de passageiro como chave se existe um identificador apropriado. Nomes podem se repetir ou variar. Chaves precisam de significado compatível; não basta terem o mesmo nome.

## Vocabulário

- **DataFrame:** tabela pandas; **Series:** uma coluna.
- **Chave:** coluna, ou combinação de colunas, usada para relacionar registros.
- **Cardinalidade:** quantas linhas podem corresponder a uma chave de cada lado.
- **Um para um:** uma linha de cada lado por chave.
- **Muitos para um:** várias linhas à esquerda podem corresponder a uma linha do cadastro à direita.
- **Chave composta:** duas ou mais colunas juntas identificam uma combinação.
- **Órfão:** um registro cuja chave não encontra o registro esperado na outra tabela.

## Símbolos e comandos

| Escrita | Significado |
|---|---|
| `=` | Guarda um resultado numa variável |
| `==` | Compara valores |
| `base["Age"]` | Seleciona uma coluna |
| `base[["Age", "Fare"]]` | Seleciona várias colunas |
| `.merge(...)` | Chama o método de junção da tabela à esquerda |
| `how="left"` | Argumento nomeado: informa o tipo de junção |
| `.shape` | Atributo: dimensões, sem parênteses |
| `.head()` | Método: mostra o começo da tabela |
| `.loc[condicao, colunas]` | Seleciona linhas e colunas |

O ponto acessa atributos e métodos. Os parênteses chamam uma operação; os argumentos são as informações entregues a ela. Não copie uma linha longa sem identificar primeiro esquerda, direita, chave e tipo de junção.

## Tipos de junção

| how | Registros mantidos | Pergunta típica |
|---|---|---|
| left | Todas as linhas da esquerda, com correspondências da direita | Quero enriquecer minha população sem excluí-la |
| inner | Somente correspondências entre os lados | Quero analisar apenas quem tem correspondência |
| outer | Registros dos dois lados, incluindo sem correspondência | Quero auditar problemas nos dois lados |

**Atenção:** left preserva as linhas da esquerda, mas pode **multiplicá-las** quando a direita tem várias correspondências. Preservar não significa garantir a mesma contagem.

`validate="one_to_one"` exige chave única nos dois lados. `validate="many_to_one"` exige chave única na direita. Essa validação não garante que todas as chaves encontraram correspondência: também precisamos conferir o indicador.

`indicator=True` cria a coluna `_merge`: `both` = correspondeu nos dois lados; `left_only` = existe apenas à esquerda; `right_only` = apenas à direita.

No pandas, chaves ausentes podem corresponder a chaves ausentes. Por isso precisamos inspecioná-las; ausência de chave não é uma identidade conhecida.

## Preparação — ler e conhecer as tabelas


### P1. Importar a biblioteca

**O que queremos:** Disponibilizar ferramentas de tabelas.

**Explique antes de executar:**

`import pandas as pd` carrega pandas com o apelido pd. Não lê arquivo e normalmente não mostra saída.

In [1]:
import pandas as pd

**O que observar e dizer na gravação:** Se a importação falhar, confira o kernel e a instalação antes de continuar.

### P. Ler passageiros

**O que queremos:** Carregar passageiros.csv.

**Explique antes de executar:**

`pd.read_csv` lê o arquivo local. `=` guarda o DataFrame na variável. A última linha usa head() para conferir as primeiras linhas.

In [2]:
passageiros = pd.read_csv("passageiros.csv")
passageiros.head()

,PassengerId,Name,Sex,Age,SibSp,Parch
0,1,"Braund, Mr. Owen Harris",male,22.0,1,0
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0
2,3,"Heikkinen, Miss. Laina",female,26.0,0,0
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0
4,5,"Allen, Mr. William Henry",male,35.0,0,0


**O que observar e dizer na gravação:** Compare os nomes das colunas com o quadro de arquivos; PassengerId aparece nas três tabelas de pessoas, mas não no cadastro de portos.

### P. Ler viagens

**O que queremos:** Carregar viagens.csv.

**Explique antes de executar:**

`pd.read_csv` lê o arquivo local. `=` guarda o DataFrame na variável. A última linha usa head() para conferir as primeiras linhas.

In [3]:
viagens = pd.read_csv("viagens.csv")
viagens.head()

,PassengerId,Pclass,Ticket,Fare,Embarked
0,1,3,A/5 21171,7.2500,S
1,2,1,PC 17599,71.2833,C
2,3,3,STON/O2. 3101282,7.9250,S
3,4,1,113803,53.1000,S
4,5,3,373450,8.0500,S


**O que observar e dizer na gravação:** Compare os nomes das colunas com o quadro de arquivos; PassengerId aparece nas três tabelas de pessoas, mas não no cadastro de portos.

### P. Ler resultados

**O que queremos:** Carregar resultados.csv.

**Explique antes de executar:**

`pd.read_csv` lê o arquivo local. `=` guarda o DataFrame na variável. A última linha usa head() para conferir as primeiras linhas.

In [4]:
resultados = pd.read_csv("resultados.csv")
resultados.head()

,PassengerId,Survived
0,1,0
1,2,1
2,3,1
3,4,1
4,5,0


**O que observar e dizer na gravação:** Compare os nomes das colunas com o quadro de arquivos; PassengerId aparece nas três tabelas de pessoas, mas não no cadastro de portos.

### P. Ler portos

**O que queremos:** Carregar portos.csv.

**Explique antes de executar:**

`pd.read_csv` lê o arquivo local. `=` guarda o DataFrame na variável. A última linha usa head() para conferir as primeiras linhas.

In [5]:
portos = pd.read_csv("portos.csv")
portos.head()

,codigo_porto,nome_porto
0,C,Cherbourg
1,Q,Queenstown
2,S,Southampton


**O que observar e dizer na gravação:** Compare os nomes das colunas com o quadro de arquivos; PassengerId aparece nas três tabelas de pessoas, mas não no cadastro de portos.

### P2. Conferir dimensões

**O que queremos:** Saber quantos registros cada arquivo tem.

**Explique antes de executar:**

`.shape` devolve (linhas, colunas). print mostra uma identificação junto do resultado.

In [6]:
print("passageiros:", passageiros.shape)
print("viagens:", viagens.shape)
print("resultados:", resultados.shape)
print("portos:", portos.shape)

passageiros: (891, 6)
viagens: (891, 5)
resultados: (891, 2)
portos: (3, 2)


**O que observar e dizer na gravação:** 891 linhas em cada tabela de passageiros; 3 linhas no cadastro de portos. O tamanho igual sozinho não prova que as chaves correspondem.

### P3. Conferir unicidade e tipos

**O que queremos:** Verificar as hipóteses das chaves.

**Explique antes de executar:**

`.is_unique` testa repetição. `.dtype` mostra o tipo da coluna. IDs 1 e texto "1" não devem ser tratados como compatíveis sem conferir os tipos.

In [7]:
print("IDs passageiros:", passageiros["PassengerId"].is_unique, passageiros["PassengerId"].dtype)
print("IDs viagens:", viagens["PassengerId"].is_unique, viagens["PassengerId"].dtype)
print("IDs resultados:", resultados["PassengerId"].is_unique, resultados["PassengerId"].dtype)
print("Códigos dos portos:", portos["codigo_porto"].is_unique)
print("Chaves ausentes em portos:", portos["codigo_porto"].isna().sum())

IDs passageiros: True int64
IDs viagens: True int64
IDs resultados: True int64
Códigos dos portos: True
Chaves ausentes em portos: 0


**O que observar e dizer na gravação:** Todas as chaves são únicas nos seus arquivos; IDs têm tipo int64; não há código ausente no cadastro de portos.

## Exercício 1 — reconstruir a base analítica

**Pergunta:** reunir passageiros, viagens e resultados, preservando todos os passageiros e uma linha por PassengerId. Mostrar nome, classe, porto e sobrevivência.

**Plano:** passageiros à esquerda → juntar viagens por ID → auditar → juntar resultados → conferir → retirar o indicador temporário.

### 1.1. Juntar viagens

**O que queremos:** Trazer dados da viagem para cada passageiro.

**Explique antes de executar:**

Em `passageiros.merge(viagens, ...)`, passageiros é a esquerda e viagens é a direita. `on="PassengerId"` indica a mesma chave em ambos. how="left" mantém os passageiros. one_to_one exige unicidade nos dois lados. indicator=True mostra a correspondência.

In [8]:
base_viagens = passageiros.merge(
    viagens, on="PassengerId", how="left",
    validate="one_to_one", indicator=True
)
base_viagens.head()

,PassengerId,Name,Sex,Age,SibSp,Parch,Pclass,Ticket,Fare,Embarked,_merge
0,1,"Braund, Mr. Owen Harris",male,22.0,1,0,3,A/5 21171,7.2500,S,both
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,1,PC 17599,71.2833,C,both
2,3,"Heikkinen, Miss. Laina",female,26.0,0,0,3,STON/O2. 3101282,7.9250,S,both
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,1,113803,53.1000,S,both
4,5,"Allen, Mr. William Henry",male,35.0,0,0,3,373450,8.0500,S,both


**O que observar e dizer na gravação:** Nome e idade vieram da esquerda; classe e tarifa, da direita. A chave comum aparece uma vez. As tabelas de entrada não foram alteradas.

### 1.2. Auditar a primeira junção

**O que queremos:** Conferir correspondências e contagem antes da próxima junção.

**Explique antes de executar:**

value_counts conta categorias do indicador. Len conta linhas; is_unique testa uma linha por ID. Só esses testes juntos começam a demonstrar que o resultado esperado foi preservado.

In [9]:
print(base_viagens["_merge"].value_counts())
print("Linhas:", len(base_viagens))
print("IDs únicos:", base_viagens["PassengerId"].is_unique)

_merge
both          891
left_only       0
right_only      0
Name: count, dtype: int64
Linhas: 891
IDs únicos: True


**O que observar e dizer na gravação:** 891 both, nenhuma ausência de correspondência; 891 linhas e IDs únicos. Com left, não haverá right_only; para enxergar sobras à direita precisaríamos de outer.

### 1.3. Juntar resultados

**O que queremos:** Acrescentar Survived à base.

**Explique antes de executar:**

drop(columns="_merge") remove o indicador antigo para que o novo merge possa criar seu próprio indicador com o mesmo nome. Depois juntamos resultados pela mesma chave.

In [10]:
base_sem_indicador = base_viagens.drop(columns="_merge")
base = base_sem_indicador.merge(
    resultados, on="PassengerId", how="left",
    validate="one_to_one", indicator=True
)
base["_merge"].value_counts()

,count
_merge,
both,891
left_only,0
right_only,0


**O que observar e dizer na gravação:** Todos os 891 têm correspondência de resultado. Survived 0 é morte registrada; ausência de correspondência seria outra situação.

### 1.4. Conferir e selecionar campos

**O que queremos:** Finalizar a base que os demais exercícios usarão.

**Explique antes de executar:**

assert interrompe se a condição não for verdadeira; quando passa, não imprime mensagem. Retiramos o indicador após conferir e selecionamos as colunas com dois colchetes.

In [11]:
assert len(base) == len(passageiros)
assert base["PassengerId"].is_unique
assert base["_merge"].eq("both").all()
base = base.drop(columns="_merge")
base[["PassengerId", "Name", "Pclass", "Embarked", "Survived"]].head()

,PassengerId,Name,Pclass,Embarked,Survived
0,1,"Braund, Mr. Owen Harris",3,S,0
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",1,C,1
2,3,"Heikkinen, Miss. Laina",3,S,1
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,S,1
4,5,"Allen, Mr. William Henry",3,S,0


**O que observar e dizer na gravação:** 891 registros e uma linha por passageiro. eq("both") equivale a comparar com ==; all() exige que todas as linhas satisfaçam a condição.

## Exercício 2 — adicionar nome do porto sem perder passageiros

**Pergunta:** as chaves têm nomes diferentes. Adicionar nome_porto, identificar ausências e comparar left com inner.

**Chaves:** Embarked na base e codigo_porto no cadastro. **Cardinalidade:** muitos passageiros para um porto. **Não inventar porto para dados ausentes.**

### 2.1. Juntar por chaves de nomes diferentes

**O que queremos:** Acrescentar o cadastro mantendo todos os passageiros.

**Explique antes de executar:**

left_on informa a coluna da esquerda; right_on, a da direita. many_to_one exige um registro por código no cadastro. Como os nomes diferem, as duas colunas de chave permanecem no resultado.

In [12]:
base_portos = base.merge(
    portos, left_on="Embarked", right_on="codigo_porto",
    how="left", validate="many_to_one", indicator=True
)
base_portos[["PassengerId", "Embarked", "codigo_porto", "nome_porto", "_merge"]].head()

,PassengerId,Embarked,codigo_porto,nome_porto,_merge
0,1,S,S,Southampton,both
1,2,C,C,Cherbourg,both
2,3,S,S,Southampton,both
3,4,S,S,Southampton,both
4,5,S,S,Southampton,both


**O que observar e dizer na gravação:** Muitos passageiros usam S, mas o cadastro tem apenas uma linha de S; essa junção não multiplica pessoas.

### 2.2. Encontrar passageiros sem correspondência

**O que queremos:** Mostrar as linhas classificadas como left_only.

**Explique antes de executar:**

loc recebe uma máscara criada com eq("left_only") e a lista de colunas. O indicador distingue ausência de correspondência de um atributo ausente em um registro que correspondeu.

In [13]:
sem_porto = base_portos.loc[
    base_portos["_merge"].eq("left_only"),
    ["PassengerId", "Embarked", "nome_porto"]
]
sem_porto

,PassengerId,Embarked,nome_porto
61,62,NaN,NaN
829,830,NaN,NaN


**O que observar e dizer na gravação:** IDs 62 e 830: Embarked está ausente. Não há código conhecido para ligar ao cadastro; não podemos escolher um porto arbitrariamente.

### 2.3. Comparar com inner

**O que queremos:** Ver o efeito de excluir registros sem correspondência.

**Explique antes de executar:**

inner mantém apenas as correspondências. Na mesma base, isso muda a população de 891 para 889; não é automaticamente um erro, mas precisa ser uma escolha consciente.

In [14]:
base_inner = base.merge(
    portos, left_on="Embarked", right_on="codigo_porto",
    how="inner", validate="many_to_one"
)
print("left:", len(base_portos))
print("inner:", len(base_inner))
print("Sem correspondência:", len(sem_porto))

left: 891
inner: 889
Sem correspondência: 2


**O que observar e dizer na gravação:** Left 891, inner 889 e 2 sem correspondência. many_to_one passou em ambos, pois valida multiplicidade, não completude.

## Exercício 3 — bilhetes compartilhados

**Pergunta:** criar uma linha por Ticket com número de passageiros no arquivo, juntar à base e analisar bilhetes com pelo menos três passageiros.

**Mudança de unidade:** o resumo tem uma linha por bilhete; a base enriquecida continua com uma linha por passageiro. Mesmo Ticket não comprova relação familiar.

### 3.1. Contar por bilhete

**O que queremos:** Construir uma tabela de referência de Ticket.

**Explique antes de executar:**

groupby("Ticket") reúne registros com o mesmo código. dropna=False inclui chaves ausentes, se existirem. agg calcula size, a quantidade de linhas; reset_index transforma a chave em coluna.

In [15]:
bilhetes = viagens.groupby("Ticket", dropna=False).agg(
    passageiros_no_train=("PassengerId", "size")
).reset_index()
bilhetes.head()

,Ticket,passageiros_no_train
0,110152,3
1,110413,3
2,110465,2
3,110564,1
4,110813,1


**O que observar e dizer na gravação:** A coluna de contagem descreve apenas passageiros nos arquivos fornecidos, não todas as pessoas que historicamente usaram o bilhete.

### 3.2. Juntar a contagem à base

**O que queremos:** Informar em cada passageiro quantos registros compartilham seu Ticket.

**Explique antes de executar:**

O Ticket se repete à esquerda, mas é único no resumo à direita: many_to_one. O total do bilhete será repetido em cada passageiro do grupo.

In [16]:
base_bilhetes = base.merge(
    bilhetes, on="Ticket", how="left", validate="many_to_one"
)
base_bilhetes[["PassengerId", "Ticket", "passageiros_no_train"]].head()

,PassengerId,Ticket,passageiros_no_train
0,1,A/5 21171,1
1,2,PC 17599,1
2,3,STON/O2. 3101282,1
3,4,113803,2
4,5,373450,1


**O que observar e dizer na gravação:** Não some passageiros_no_train depois do merge como se fossem pessoas diferentes: é um tamanho de grupo repetido por linha.

### 3.3. Filtrar os bilhetes desejados

**O que queremos:** Selecionar passageiros com Ticket presente pelo menos três vezes.

**Explique antes de executar:**

>= 3 inclui exatamente três. loc filtra as linhas da base enriquecida. Ainda estamos vendo passageiros, não uma linha por bilhete.

In [17]:
compartilhados = base_bilhetes.loc[base_bilhetes["passageiros_no_train"] >= 3]
compartilhados[["PassengerId", "Ticket", "passageiros_no_train"]].head()

,PassengerId,Ticket,passageiros_no_train
7,8,349909,4
8,9,347742,3
13,14,347082,7
16,17,382652,5
24,25,349909,4


**O que observar e dizer na gravação:** Repare nos bilhetes repetidos. Para responder por bilhete, precisamos agrupar novamente.

### 3.4. Resumir sobrevivência por bilhete

**O que queremos:** Voltar à unidade de uma linha por Ticket.

**Explique antes de executar:**

size conta passageiros; sum de Survived conta sobreviventes; mean do indicador 0/1 calcula proporção. Multiplicar por 100 cria percentual. sort_values aceita duas colunas e uma direção para cada uma.

In [18]:
resumo_bilhetes = compartilhados.groupby("Ticket", dropna=False).agg(
    passageiros=("PassengerId", "size"),
    sobreviventes=("Survived", "sum"),
    taxa_sobrevivencia=("Survived", "mean")
).reset_index()
resumo_bilhetes["taxa_percentual"] = resumo_bilhetes["taxa_sobrevivencia"] * 100
resumo_bilhetes.sort_values(["passageiros", "Ticket"], ascending=[False, True]).head(10)

,Ticket,passageiros,sobreviventes,taxa_sobrevivencia,taxa_percentual
5,1601,7,5,0.714286,71.428571
17,347082,7,0,0.000000,0.000000
29,CA. 2343,7,0,0.000000,0.000000
14,3101295,6,0,0.000000,0.000000
18,347088,6,0,0.000000,0.000000
28,CA 2144,6,0,0.000000,0.000000
24,382652,5,0,0.000000,0.000000
37,S.O.C. 14879,5,0,0.000000,0.000000
2,113760,4,4,1.000000,100.000000
3,113781,4,2,0.500000,50.000000


**O que observar e dizer na gravação:** Contamos linhas reais de passageiros; não somamos a contagem repetida. Taxas com poucos passageiros devem ser lidas junto do tamanho do grupo.

## Exercício 4 — comparar tarifa com a mediana de seu grupo

**Pergunta:** calcular mediana por Pclass e Sex, juntar usando as duas chaves e selecionar tarifas acima da mediana.

Mediana é o valor central dos valores ordenados, ou a média dos dois centrais quando a quantidade é par. Aqui o grupo é definido pela combinação classe + sexo.

### 4.1. Construir a referência por grupo

**O que queremos:** Ter uma mediana por combinação de classe e sexo.

**Explique antes de executar:**

groupby recebe a lista de duas colunas. median ignora tarifas ausentes. O nome mediana_tarifa será a coluna de saída.

In [19]:
medianas = base.groupby(["Pclass", "Sex"], dropna=False).agg(
    mediana_tarifa=("Fare", "median")
).reset_index()
medianas

,Pclass,Sex,mediana_tarifa
0,1,female,82.66455
1,1,male,41.26250
2,2,female,22.00000
3,2,male,13.00000
4,3,female,12.47500
5,3,male,7.92500


**O que observar e dizer na gravação:** Há seis combinações. Pclass sozinha aparece duas vezes para cada classe: não identifica a linha do resumo.

### 4.2. Juntar com chave composta

**O que queremos:** Dar a cada passageiro a mediana da combinação correta.

**Explique antes de executar:**

on=["Pclass", "Sex"] exige correspondência simultânea nas duas chaves. many_to_one exige unicidade dessa combinação à direita. Usar apenas Pclass faria duas correspondências por passageiro e seria rejeitado por essa validação.

In [20]:
comparacao = base.merge(
    medianas, on=["Pclass", "Sex"],
    how="left", validate="many_to_one"
)
comparacao[["PassengerId", "Pclass", "Sex", "Fare", "mediana_tarifa"]].head()

,PassengerId,Pclass,Sex,Fare,mediana_tarifa
0,1,3,male,7.2500,7.92500
1,2,1,female,71.2833,82.66455
2,3,3,female,7.9250,12.47500
3,4,1,female,53.1000,82.66455
4,5,3,male,8.0500,7.92500


**O que observar e dizer na gravação:** Cada passageiro recebe a referência de seu grupo. A mediana é repetida dentro da combinação; não é a mediana geral.

### 4.3. Calcular e filtrar diferenças

**O que queremos:** Mostrar apenas tarifas acima da referência.

**Explique antes de executar:**

A subtração entre colunas acontece linha a linha. Compare Fare com mediana_tarifa e selecione as colunas pedidas. Uma comparação com NaN não confirma que a tarifa é maior.

In [21]:
comparacao["diferenca"] = comparacao["Fare"] - comparacao["mediana_tarifa"]
acima = comparacao.loc[
    comparacao["Fare"] > comparacao["mediana_tarifa"],
    ["PassengerId", "Pclass", "Sex", "Fare", "mediana_tarifa", "diferenca"]
]
acima.sort_values("diferenca", ascending=False).head(10)

,PassengerId,Pclass,Sex,Fare,mediana_tarifa,diferenca
679,680,1,male,512.3292,41.26250,471.06670
737,738,1,male,512.3292,41.26250,471.06670
258,259,1,female,512.3292,82.66455,429.66465
438,439,1,male,263.0000,41.26250,221.73750
27,28,1,male,263.0000,41.26250,221.73750
118,119,1,male,247.5208,41.26250,206.25830
557,558,1,male,227.5250,41.26250,186.26250
527,528,1,male,221.7792,41.26250,180.51670
341,342,1,female,263.0000,82.66455,180.33545
88,89,1,female,263.0000,82.66455,180.33545


**O que observar e dizer na gravação:** Todas as diferenças mostradas são positivas. Fare é a tarifa registrada: esta atividade não infere um preço individual preciso em bilhetes compartilhados.

## Exercício 5 — auditoria com resultado ausente e chave órfã

**Pergunta:** simular dois problemas numa cópia: retirar o resultado do primeiro passageiro e adicionar um ID inexistente na base. Usar outer para enxergar ambos.

É uma simulação didática. Os arquivos originais não serão modificados.

### 5.1. Escolher os IDs da simulação

**O que queremos:** Definir qual resultado remover e qual chave adicionar.

**Explique antes de executar:**

iloc[0] seleciona a primeira linha pela posição. max()+1 cria um ID maior que todos os existentes neste exercício. int converte o valor para inteiro Python.

In [22]:
id_removido = int(resultados.iloc[0]["PassengerId"])
id_orfao = int(base["PassengerId"].max()) + 1
print("Resultado removido:", id_removido)
print("ID órfão adicionado:", id_orfao)

Resultado removido: 1
ID órfão adicionado: 892


**O que observar e dizer na gravação:** Retiraremos o resultado de PassengerId 1 e adicionaremos ID 892, sem pessoa correspondente.

### 5.2. Criar os resultados alterados

**O que queremos:** Montar uma tabela nova com os dois problemas.

**Explique antes de executar:**

!= seleciona IDs diferentes do removido. pd.DataFrame cria a linha artificial. pd.concat empilha tabelas verticalmente: não procura chave, portanto é diferente de merge. ignore_index=True recria o índice de linhas.

In [23]:
sem_primeiro = resultados.loc[resultados["PassengerId"] != id_removido]
linha_orfa = pd.DataFrame({"PassengerId": [id_orfao], "Survived": [1]})
resultados_auditoria = pd.concat([sem_primeiro, linha_orfa], ignore_index=True)
resultados_auditoria.tail()

,PassengerId,Survived
886,888,1
887,889,0
888,890,1
889,891,0
890,892,1


**O que observar e dizer na gravação:** A contagem continua 891, apesar dos problemas. Por isso comparar somente o tamanho dos arquivos não é suficiente.

### 5.3. Fazer outer e examinar o indicador

**O que queremos:** Enxergar problemas dos dois lados da junção.

**Explique antes de executar:**

outer mantém a união dos IDs. one_to_one continua válido porque os IDs não se repetem; indicator mostra quais não encontraram par.

In [24]:
auditoria = passageiros.merge(
    resultados_auditoria, on="PassengerId", how="outer",
    validate="one_to_one", indicator=True
)
auditoria.loc[auditoria["_merge"] != "both", ["PassengerId", "Name", "Survived", "_merge"]]

,PassengerId,Name,Survived,_merge
0,1,"Braund, Mr. Owen Harris",NaN,left_only
891,892,NaN,1.0,right_only


**O que observar e dizer na gravação:** ID 1 é left_only, com resultado ausente; ID 892 é right_only, com nome ausente. A união tem 892 IDs. Ausência de resultado não significa Survived=0: preencher com zero inventaria morte registrada.

## Exercício 6 — um merge que multiplica passageiros

**Pergunta:** duplicar artificialmente o primeiro porto, testar a validação, observar a inflação sem validação e corrigir.

Imagine duas linhas de cadastro para C. Cada passageiro de C encontrará as duas; a junção gerará duas linhas para essa pessoa.

### 6.1. Duplicar o primeiro cadastro

**O que queremos:** Criar uma duplicação exata e controlada.

**Explique antes de executar:**

iloc[[0]] devolve a primeira linha como tabela; colchetes duplos mantêm o formato DataFrame. concat a acrescenta ao cadastro, que passa de três para quatro linhas.

In [25]:
portos_duplicados = pd.concat([portos, portos.iloc[[0]]], ignore_index=True)
portos_duplicados

,codigo_porto,nome_porto
0,C,Cherbourg
1,Q,Queenstown
2,S,Southampton
3,C,Cherbourg


**O que observar e dizer na gravação:** C/Cherbourg aparece duas vezes. A duplicação foi criada apenas para demonstrar o problema.

### 6.2. Capturar o erro de cardinalidade

**O que queremos:** Ver many_to_one impedir uma junção incompatível.

**Explique antes de executar:**

try tenta a operação; except captura especificamente MergeError. Se não houver erro, o else acusa uma falha no cenário esperado. Não escondemos qualquer erro com uma captura genérica.

In [26]:
try:
    base.merge(
        portos_duplicados, left_on="Embarked", right_on="codigo_porto",
        how="left", validate="many_to_one"
    )
except pd.errors.MergeError as erro:
    print("Erro esperado:", erro)
else:
    raise AssertionError("A chave duplicada deveria ser detectada.")

Erro esperado: Merge keys are not unique in right dataset; not a many-to-one merge


**O que observar e dizer na gravação:** A mensagem indica que as chaves da direita não são únicas. O notebook continua porque o erro intencional foi capturado.

### 6.3. Executar sem validação para investigar

**O que queremos:** Medir o efeito da duplicação.

**Explique antes de executar:**

Nesta demonstração retiramos validate para enxergar o problema, não como correção. len(inflada)-len(base) mede as linhas extras.

In [27]:
inflada = base.merge(
    portos_duplicados, left_on="Embarked", right_on="codigo_porto", how="left"
)
print("Antes:", len(base))
print("Depois:", len(inflada))
print("Linhas extras:", len(inflada) - len(base))

Antes: 891
Depois: 1059
Linhas extras: 168


**O que observar e dizer na gravação:** 891 → 1059: 168 linhas extras, uma para cada passageiro cujo Embarked é C. Uma soma de sobreviventes nessa tabela poderia ser distorcida.

### 6.4. Mostrar IDs repetidos

**O que queremos:** Identificar quem foi multiplicado.

**Explique antes de executar:**

duplicated(keep=False) marca todas as ocorrências de IDs repetidos, não somente a segunda. loc mostra campos para investigar.

In [28]:
ids_repetidos = inflada["PassengerId"].duplicated(keep=False)
inflada.loc[ids_repetidos, ["PassengerId", "Embarked", "nome_porto"]].head(10)

,PassengerId,Embarked,nome_porto
1,2,C,Cherbourg
2,2,C,Cherbourg
10,10,C,Cherbourg
11,10,C,Cherbourg
21,20,C,Cherbourg
22,20,C,Cherbourg
29,27,C,Cherbourg
30,27,C,Cherbourg
34,31,C,Cherbourg
35,31,C,Cherbourg


**O que observar e dizer na gravação:** Os mesmos IDs aparecem em duas linhas iguais de porto. Um passageiro continua sendo uma pessoa, apesar de ocupar duas linhas no resultado.

### 6.5. Corrigir a origem e refazer

**O que queremos:** Remover a duplicação exata e retomar a validação.

**Explique antes de executar:**

drop_duplicates() sem subset remove linhas inteiramente iguais. É apropriado aqui porque a cópia artificial é exata. Se o mesmo código tiver nomes ou versões diferentes, é preciso decidir qual registro é válido; não basta escolher o primeiro.

In [29]:
portos_corrigidos = portos_duplicados.drop_duplicates()
assert portos_corrigidos["codigo_porto"].is_unique
corrigida = base.merge(
    portos_corrigidos, left_on="Embarked", right_on="codigo_porto",
    how="left", validate="many_to_one"
)
print("Linhas corrigidas:", len(corrigida))
print("Uma linha por passageiro:", corrigida["PassengerId"].is_unique)

Linhas corrigidas: 891
Uma linha por passageiro: True


**O que observar e dizer na gravação:** Voltamos a 891 e True para unicidade. Remover duplicatas do resultado final sem entender a origem poderia esconder o problema.

## Desafio final — relatório por porto e classe

**Pergunta:** apresentar passageiros, sobreviventes, taxa e mediana de idade por nome_porto e Pclass; incluir idades conhecidas e preservar quem não tem porto.

O texto Não informado será um rótulo de apresentação, não um porto imputado.

### D1. Preparar o rótulo do relatório

**O que queremos:** Enriquecer a base e criar uma coluna só para exibição.

**Explique antes de executar:**

Faça o merge com o cadastro original e confira a unicidade. Crie porto_relatorio preenchendo apenas o rótulo, preservando nome_porto e Embarked originais.

In [30]:
relatorio_base = base.merge(
    portos, left_on="Embarked", right_on="codigo_porto",
    how="left", validate="many_to_one"
)
assert relatorio_base["PassengerId"].is_unique
relatorio_base["porto_relatorio"] = relatorio_base["nome_porto"].fillna("Não informado")
relatorio_base.loc[relatorio_base["nome_porto"].isna(), ["PassengerId", "Embarked", "nome_porto", "porto_relatorio"]]

,PassengerId,Embarked,nome_porto,porto_relatorio
61,62,NaN,NaN,Não informado
829,830,NaN,NaN,Não informado


**O que observar e dizer na gravação:** Os dois passageiros sem porto continuam presentes. O rótulo não altera os códigos ausentes.

### D2. Construir as métricas

**O que queremos:** Resumir por porto e classe.

**Explique antes de executar:**

size conta linhas; sum do indicador conta sobreviventes; mean calcula proporção; median em Age ignora ausências; count em Age conta somente idades conhecidas. dropna=False preserva chaves ausentes se aparecerem.

In [31]:
relatorio = relatorio_base.groupby(["porto_relatorio", "Pclass"], dropna=False).agg(
    passageiros=("PassengerId", "size"),
    sobreviventes=("Survived", "sum"),
    taxa_sobrevivencia=("Survived", "mean"),
    mediana_idade=("Age", "median"),
    idades_conhecidas=("Age", "count")
).reset_index()
relatorio["taxa_percentual"] = relatorio["taxa_sobrevivencia"] * 100
relatorio

,porto_relatorio,Pclass,passageiros,sobreviventes,taxa_sobrevivencia,mediana_idade,idades_conhecidas,taxa_percentual
0,Cherbourg,1,85,59,0.694118,36.5,74,69.411765
1,Cherbourg,2,17,9,0.529412,25.0,15,52.941176
2,Cherbourg,3,66,25,0.378788,20.0,41,37.878788
3,Não informado,1,2,2,1.000000,50.0,2,100.000000
4,Queenstown,1,2,1,0.500000,38.5,2,50.000000
5,Queenstown,2,3,2,0.666667,43.5,2,66.666667
6,Queenstown,3,72,27,0.375000,21.5,24,37.500000
7,Southampton,1,127,74,0.582677,37.0,108,58.267717
8,Southampton,2,164,76,0.463415,30.0,156,46.341463
9,Southampton,3,353,67,0.189802,25.0,290,18.980170


**O que observar e dizer na gravação:** A mediana descreve somente as idades conhecidas; a diferença entre passageiros e idades_conhecidas mostra a incompletude do grupo.

### D3. Conferir e interpretar

**O que queremos:** Garantir que o relatório incluiu todos os passageiros.

**Explique antes de executar:**

Some as contagens dos grupos e compare com a base. Inclua os tamanhos na interpretação, pois percentuais de grupos pequenos variam com poucas pessoas.

In [32]:
print("Passageiros no relatório:", relatorio["passageiros"].sum())
print("Sobreviventes:", relatorio["sobreviventes"].sum())
relatorio.loc[relatorio["porto_relatorio"] == "Não informado"]

Passageiros no relatório: 891
Sobreviventes: 342


,porto_relatorio,Pclass,passageiros,sobreviventes,taxa_sobrevivencia,mediana_idade,idades_conhecidas,taxa_percentual
3,Não informado,1,2,2,1.0,50.0,2,100.0


**O que observar e dizer na gravação:** 891 passageiros, 342 sobreviventes. O grupo Não informado tem 2 pessoas, ambas sobreviventes: 100% nesse grupo minúsculo não demonstra um padrão geral.

## Fechamento para a gravação

“Antes de juntar, preciso saber o que cada linha representa, qual é a chave e quantas correspondências espero. Depois de juntar, confiro quantidade, unicidade e correspondências. Um código que executou sem erro ainda pode produzir uma tabela errada para a pergunta.”

**Perguntas de revisão, com respostas:**

1. left impede duplicações? **Não; mantém a esquerda e pode multiplicá-la se houver várias correspondências à direita.**
2. many_to_one exige o quê? **Chave única na tabela à direita.**
3. Qual junção enxerga órfãos nos dois lados? **outer, usando indicator para identificá-los.**
4. Por que usar Pclass e Sex juntas? **Porque a mediana foi calculada por essa combinação.**
5. Posso somar o tamanho do bilhete repetido em cada passageiro? **Não para contar pessoas: o tamanho do grupo está repetido.**
6. Ausência de resultado significa morte? **Não; significa que o resultado não foi obtido naquele cruzamento.**
7. concat e merge fazem a mesma coisa? **Não; concat aqui empilha linhas, enquanto merge relaciona tabelas por chaves.**

## Conferências finais
`assert` interrompe se uma hipótese não for verdadeira. As verificações abaixo resumem o que foi conferido com os arquivos enviados.


In [33]:
assert len(base) == len(passageiros) == 891
assert base["PassengerId"].is_unique
assert base["Survived"].sum() == 342
assert len(base_portos) == 891 and len(base_inner) == 889
assert set(sem_porto["PassengerId"]) == {62, 830}
assert len(base_bilhetes) == len(base)
assert (resumo_bilhetes["sobreviventes"] <= resumo_bilhetes["passageiros"]).all()
assert len(comparacao) == len(base)
assert acima["diferenca"].gt(0).all()
assert auditoria["_merge"].eq("left_only").sum() == 1
assert auditoria["_merge"].eq("right_only").sum() == 1
assert auditoria.loc[auditoria["_merge"].eq("left_only"), "Survived"].isna().all()
assert len(inflada) == 1059
assert len(inflada) - len(base) == base["Embarked"].eq(portos.iloc[0]["codigo_porto"]).sum()
assert len(corrigida) == 891 and corrigida["PassengerId"].is_unique
assert relatorio["passageiros"].sum() == 891
assert relatorio["sobreviventes"].sum() == 342
print("Todas as conferências passaram.")

Todas as conferências passaram.


## Fonte e limites
Adaptado do notebook `titanic_merge_tabelas_nivel4(1).ipynb` enviado pela professora e executado com os quatro CSVs anexados. As tabelas de passageiros, viagens e resultados foram separadas do conjunto Titanic; portos é um cadastro didático. As alterações dos exercícios 5 e 6 são simulações em memória.

As contagens descrevem os 891 passageiros presentes nestes arquivos. Bilhetes compartilhados não identificam necessariamente famílias; diferenças entre grupos não estabelecem causalidade. Esta aula não treina modelo preditivo nem consulta fontes externas.